# Swiss Commutes: deployment audit

Rechecks all 16 locations after deployment. The companion deployment receipt ties the audited files and counts to the live website. The statistical assumptions and remaining coverage gaps still apply.


In [1]:
from pathlib import Path
import json
root = Path.cwd()
if not (root / 'audit.json').exists(): root = root / 'audits/2026-09-06-deployed'
audit = json.loads((root / 'audit.json').read_text())
findings = json.loads((root / 'findings.json').read_text())
failures = [r for r in audit['checks'] if r['status']=='fail']
print('Snapshot:', audit['generatedAt'])
print('Locations:', len(audit['cityRows']), 'Corridors:', len(audit['routes']))
print('Checks:', len(audit['checks']), 'Failures:', len(failures))
for r in failures: print(r)


Snapshot: 2026-09-06T20:57:52.917Z
Locations: 16 Corridors: 24673
Checks: 716 Failures: 0


## Geneva before and after

These are model snapshots, not a change in real traffic. Mapped French totals remain constant; routing coverage changes. Vaud mode totals use historical directional survey shares, with other modes unspecified.


In [2]:
before = json.loads((root.parent / "2026-09-06-fixed/audit.json").read_text())
def cohort(snapshot, group, mode):
    return [r for r in snapshot["routes"] if r["city"] == "geneva" and r["group"] == group and r["mode"] == mode]
def total(rows, routed=False):
    return sum(r["commuters"] for r in rows if not routed or r["geometry"] != "missing")
old = cohort(before, "foreignInbound", "transit")
new = cohort(audit, "foreignInbound", "transit")
assert total(old) == total(new) == 18812
assert total(new, True) > total(old, True)
print("French PT: mapped", total(new), "routed before", total(old, True), "after", total(new, True))
for mode in ["car", "transit", "soft", "unknown"]:
    print("Vaud incoming", mode, total(cohort(before, "swissInbound", mode)), "→", total(cohort(audit, "swissInbound", mode)))
assert not cohort(audit, "swissInbound", "soft")
assert total(cohort(audit, "swissInbound", "unknown")) > 0
assert total(cohort(audit, "swissInbound", "unknown"), True) == 0
source = audit["originalSources"]["genevaVaudModes"]
assert source["inbound"] == {"car":9389,"transit":13437,"unknown":858,"total":23684}
assert source["outbound"] == {"car":2561,"transit":4419,"unknown":521,"total":7501}
scheduled = [r for r in new if r["geometry"] == "timetable"]
print("Timetabled French origins:", len(set(r["origin"] for r in scheduled)))
print("French commuters on timetable routes:", total(scheduled))


French PT: mapped 18812 routed before 3083 after 16081
Vaud incoming car 9516 → 8469
Vaud incoming transit 7047 → 12103
Vaud incoming soft 4738 → 0
Vaud incoming unknown 0 → 729
Timetabled French origins: 76
French commuters on timetable routes: 16081


## Disposition of the original findings

Open and partly corrected findings remain material even when numerical checks pass. The neighbouring 2026-09-06 directory preserves the baseline.

In [3]:
for r in findings: print(r['id'], '|', r['status'], '|', r['title'])

F01 | Partly corrected | Direction-specific mode shares
F02 | Corrected | Shared journeys agree across pages
F03 | Corrected | Motorcycles are disclosed
F04 | Corrected | No-journey records excluded
F05 | Partly corrected | Active-mode residual
F06 | Corrected | Geneva uses a consistent Swiss scope
F07 | Corrected | Geneva workplaces retained
F08 | Partly corrected | Coverage disclosed
F09 | Partly corrected | Unrouted active estimates removed from chart
F10 | Corrected | One journey clock
F11 | Open limitation | Illustrative working day
F12 | Partly corrected | Timetable journeys and rail fallback
F13 | Open limitation | Foreign origins outside Geneva
F14 | Partly corrected | Vintages and exclusions exposed
F15 | Corrected | Road endpoint review
F16 | Partly corrected | Reproducible inputs and checks


## Reciprocal journeys and timing

Compare shared Swiss municipal pairs across pages and independent arrival accounting on the same routed car cohort. Geneva has a different geography and vintage and is excluded from reciprocal municipal comparisons.

In [4]:
assert all(r['delta']==0 for r in audit['reciprocal'])
assert all(r['populationCarDisagreement']==0 for r in audit['cityRows'])
assert audit['french']['mismatches']==0 and not audit['french']['omitted']
print('Shared OD comparisons:', len(audit['reciprocal']))
print('French workplace/mode rows:', audit['french']['comparedRows'])
print('Excluded no-journey survey weight:', audit['french']['byOriginalMode']['1'])


Shared OD comparisons: 160
French workplace/mode rows: 3130
Excluded no-journey survey weight: 157.64459489058729


## Routing coverage

The denominator is the mapped estimate, after geographic selection. Coverage is completeness, not accuracy. Long-distance active estimates remain in the evidence but are excluded from the chart.

In [5]:
for city in audit['cityRows']:
    parts=[]
    for mode in ['car','transit','soft','unknown']:
        rows=[r for r in audit['modes'] if r['city']==city['city'] and r['mode']==mode]
        mapped=sum(r['people'] for r in rows); routed=sum(r['covered'] for r in rows)
        parts.append(f'{mode}: {routed:,}/{mapped:,} ({routed/mapped:.1%})' if mapped else f'{mode}: no estimates')
    print(city['name'] + ' | ' + ' | '.join(parts))


Zürich | car: 82,348/86,144 (95.6%) | transit: 162,472/203,572 (79.8%) | soft: 5,430/8,112 (66.9%) | unknown: no estimates
Geneva | car: 98,836/103,673 (95.3%) | transit: 31,201/34,167 (91.3%) | soft: 6,227/6,796 (91.6%) | unknown: 0/1,079 (0.0%)
Basel | car: 29,743/30,953 (96.1%) | transit: 28,266/63,047 (44.8%) | soft: 21,822/24,445 (89.3%) | unknown: no estimates
Lausanne | car: 35,448/37,247 (95.2%) | transit: 22,474/36,676 (61.3%) | soft: 13,535/19,580 (69.1%) | unknown: 0/242 (0.0%)
Bern | car: 38,785/40,577 (95.6%) | transit: 53,611/69,146 (77.5%) | soft: 5,287/7,951 (66.5%) | unknown: no estimates
Winterthur | car: 26,464/27,758 (95.3%) | transit: 30,051/35,859 (83.8%) | soft: 1,926/2,600 (74.1%) | unknown: no estimates
Lucerne | car: 24,444/25,566 (95.6%) | transit: 22,625/27,681 (81.7%) | soft: 3,734/4,894 (76.3%) | unknown: no estimates
St. Gallen | car: 31,637/33,256 (95.1%) | transit: 18,823/25,194 (74.7%) | soft: 954/1,620 (58.9%) | unknown: no estimates
Lugano | car: 20,

## Largest missing routes

These are missing geometries for estimated categories, not confirmed missing train services. Filter the complete corridors.csv for other views; retain the category, scope and source limitations.

In [6]:
missing=sorted((r for r in audit['routes'] if r['geometry']=='missing'),key=lambda r:r['commuters'],reverse=True)
for r in missing[:15]: print(r['city'], r['originName'], '→', r['targetName'], r['mode'], r['commuters'])
assert not failures, 'Review failed checks before treating the build as numerically reconciled'


basel Saint-Louis → Basel transit 2746
basel Binningen → Basel transit 2242
basel Reinach (BL) → Basel transit 1805
basel Huningue → Basel transit 1685
zurich Volketswil → Zürich transit 1574
basel Grenzach-Wyhlen → Basel transit 1496
basel Birsfelden → Basel transit 1420
zurich Fällanden → Zürich transit 1344
basel Oberwil (BL) → Basel transit 1262
bern Wohlen bei Bern → Bern transit 1236
zurich Oberengstringen → Zürich transit 1184
basel Basel → Allschwil transit 1100
basel Therwil → Basel transit 1027
zurich Herrliberg → Zürich transit 1007
zurich Wettswil am Albis → Zürich transit 896


## Live build verification

Remote file hashes must match exactly. Public HTML may contain Cloudflare’s injected analytics beacon; only that beacon is excluded from the public-content comparison. All-mode and public-transport counters are checked against the audit at 07:45.


In [7]:
deployment = json.loads((root / "deployment-verification.json").read_text())
http = json.loads((root / "http-verification.json").read_text())
previous = json.loads((root.parent / "2026-09-06-priorities/audit.json").read_text())
assert deployment["status"] == "verified"
assert deployment["files"] ["checked"] == 164 and deployment["files"]["mismatches"] == 0
assert http["passed"] == http["checked"] == 17
assert all(r["sha256"] == r["expectedSha256"] for r in http["pages"])
assert audit["routes"] == previous["routes"] and audit["hourly"] == previous["hourly"]
for result in deployment["browser"]["results"]:
    expected = next(r for r in audit["hourly"] if r["city"] == result["city"] and r["minute"] == 465)
    assert abs(result["all"][0] - sum(expected[k] for k in ["carNow", "transitNow", "activeNow"])) <= 1
    assert result["all"][1] == expected["vsDailyAverage"]
    assert result["transit"][0] == expected["transitNow"]
    assert all(result["defaultModes"]) and result["none"] == [0,0,0]
    assert result["rewrittenScripts"] == 0 and not result["horizontalOverflow"]
    print(result["city"], "live counts and filters agree")
assert not deployment["browser"]["applicationExceptions"]
print("164 origin files verified; 17 public pages match; commuter estimates unchanged")


geneva live counts and filters agree
lausanne live counts and filters agree
zurich live counts and filters agree
164 origin files verified; 17 public pages match; commuter estimates unchanged
